# TEST SINGLE AUGMENTATION

In [ ]:
from pathlib import Path
import os
import numpy as np
import torch
import io
from PIL import Image
import open3d as o3d

from emma import EmmaDataset, EmmaSearcher, SearchQuery

from transformers import DepthProImageProcessor, DepthProForDepthEstimation

In [ ]:
# Check CUDA Availability

if torch.cuda.is_available():
    print("Ready")
    device = torch.device('cuda')
else:
    print("Not Ready")
    device = torch.device('cpu')

## > DATASET

In [ ]:
dataset_root = None  # e.g. Path("/path/to/EMMa")

if dataset_root is None:
    raise ValueError("Please set `dataset_root` to the path of the EMMa dataset.")

dataset_root = Path(dataset_root)

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_root}")

In [ ]:
# INIT DATASET
ds = EmmaDataset(dataset_root)

In [ ]:
# QUERY
se = EmmaSearcher(ds)
query = SearchQuery(splits=["Test"], require_listed=["mass", "category"], require_present=["package_dimensions"], quantity_range=[1,1], limit=10)
asins = se.query_asins(query)
print(len(asins))

In [ ]:
# SELECT A SAMPLE
asin = asins[9]
rec = ds.get_record(asin=asin)
title = ds.get_title(rec=rec)
hashes = ds.get_image_hashes(rec=rec)
images = ds.get_images(image_hashes=hashes)
categories = ds.get_category_names(rec=rec)
package_dimensions = ds.get_package_dimensions(rec=rec)
package_dimensions = np.squeeze(np.asarray(ds.get_package_dimensions(rec=rec), dtype=np.float32)) * 2.54 / 100.0

rgb = Image.open(io.BytesIO(images[hashes[0]])).convert("RGB")
rgb_numpy = np.array(rgb).astype(float) / 255
input_image = [rgb]

print("ASIN: ", asin)
print("Title: ", title)
print("Category Path: ", categories)
print("Deepest Category: ", categories[-1])
print("Package Dimensions: ", package_dimensions)
rgb.show()

## > MODEL

In [ ]:
depth_pro_root = None  # e.g. Path("/path/to/Depth-Pro")

if depth_pro_root is None:
    raise ValueError("Please set `depth_pro_root` to the path of the Depth Pro model.")

depth_pro_root = Path(depth_pro_root)

if not depth_pro_root.exists():
    raise FileNotFoundError(f"Model path does not exist: {depth_pro_root}")

In [ ]:
image_processor = DepthProImageProcessor.from_pretrained(depth_pro_root)
model = DepthProForDepthEstimation.from_pretrained(depth_pro_root, torch_dtype=torch.float16).to(device)

## > INFERENCE

In [ ]:
inputs = image_processor(images=rgb, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

post_processed_output = image_processor.post_process_depth_estimation(
    outputs, target_sizes=[(rgb.height, rgb.width)],
)

field_of_view = post_processed_output[0]["field_of_view"].detach().cpu().numpy()
focal_length = post_processed_output[0]["focal_length"].detach().cpu().numpy()
depth = post_processed_output[0]["predicted_depth"].detach().cpu().numpy()

## > VISUALIZE

In [ ]:
depth_img = (depth - depth.min()) / (depth.max() - depth.min())
depth_img = depth_img * 255.
depth_img = Image.fromarray(depth_img.astype("uint8"))

print("field_of_view: ", field_of_view)
print("focal_length: ", focal_length)
depth_img.show()

In [ ]:
def ensure_image_format(image):
    """Ensure image is in the correct format (H, W, 3) and properly scaled."""
    if isinstance(image, torch.Tensor):
        image = image.cpu().detach().numpy()
    
    if image.shape[0] == 3:  # If in (3, H, W) format
        image = np.transpose(image, (1, 2, 0))
    
    if image.min() < 0 or image.max() > 1:
        image = (image + 1) / 2  # Normalize to [0,1]
    
    return np.clip(image, 0, 1)

def create_point_cloud(depth_map, image, focal_px, scale_ratio=1.0):
    H, W = depth_map.shape
    image = ensure_image_format(image)
    
    x_grid, y_grid = np.meshgrid(np.arange(W), np.arange(H))
    cx, cy = W / 2, H / 2

    Z = depth_map * scale_ratio
    X = (x_grid - cx) * Z / focal_px
    Y = (y_grid - cy) * Z / focal_px

    point_map = np.stack((X, Y, Z), axis=-1)

    point_cloud = o3d.geometry.PointCloud()
    point_cloud.points = o3d.utility.Vector3dVector(point_map.reshape(-1, 3))
    point_cloud.colors = o3d.utility.Vector3dVector(image.reshape(-1, 3))

    valid_points = Z.flatten() > 0
    point_cloud.points = o3d.utility.Vector3dVector(np.asarray(point_cloud.points)[valid_points])
    point_cloud.colors = o3d.utility.Vector3dVector(np.asarray(point_cloud.colors)[valid_points])

    return point_cloud

def save_point_cloud(point_cloud, name):
    output_dir = "generated_pointclouds"
    os.makedirs(output_dir, exist_ok=True)  # Ensure directory exists

    filename = os.path.basename(name).split('.')[0] + ".ply"
    output_path = os.path.join(output_dir, filename)
    o3d.io.write_point_cloud(output_path, point_cloud)
    print(f"Saved: {output_path}")

In [ ]:
point_clouds = create_point_cloud(depth, rgb_numpy, focal_length)

In [ ]:
save_point_cloud(point_clouds, "scene.ply")